<a href="https://colab.research.google.com/github/mrdbourke/pytorch-deep-learning/blob/main/extras/exercises/07_pytorch_experiment_tracking_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07. PyTorch 实验跟踪练习模板

欢迎来到 07 章 PyTorch 实验跟踪练习模板 notebook。

> **注意：** 每道练习通常不止一种解法，本 notebook 仅展示其中一种可行方案。

## 资源

1. 这些练习/解答基于 Zero to Mastery 的 Learn PyTorch for Deep Learning 课程中 [07. PyTorch Transfer Learning](https://www.learnpytorch.io/07_pytorch_experiment_tracking/) 章节。
2. 可观看 YouTube 上的[完整解题实战（含报错与调试过程）](https://youtu.be/cO_r2FYcAjU)。
3. 更多解答见课程 GitHub 的 [solutions 目录](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/extras/solutions)。

> **注意：** 本 notebook 第一部分用于准备练习所需的辅助函数与数据集。练习从标题“Exercise 1: ...”开始。

### 获取常用导入与辅助函数

我们需要确保环境中 `torch` 版本为 1.12+，`torchvision` 版本为 0.13+。

In [ ]:
# 为了兼容本 notebook 中更新后的 API，需要 torch 1.12+ 和 torchvision 0.13+
try:
    import torch
    import torchvision
    assert int(torch.__version__.split(".")[1]) >= 12, "torch version should be 1.12+"
    assert int(torchvision.__version__.split(".")[1]) >= 13, "torchvision version should be 0.13+"
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")
except:
    print(f"[INFO] torch/torchvision versions not as required, installing nightly versions.")
    !pip3 install -U --pre torch torchvision --extra-index-url https://download.pytorch.org/whl/nightly/cu113
    import torch
    import torchvision
    print(f"torch version: {torch.__version__}")
    print(f"torchvision version: {torchvision.__version__}")

torch version: 1.13.0.dev20220622+cu113
torchvision version: 0.14.0.dev20220622+cu113


In [ ]:
 # 确保有可用 GPU
 device = "cuda" if torch.cuda.is_available() else "cpu"
 device

'cuda'

In [ ]:
# 常用导入
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# 尝试导入 torchinfo，如果不可用则安装
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# 尝试导入 going_modular 目录，不存在则从 GitHub 下载
try:
    from going_modular.going_modular import data_setup, engine
except:
    # 获取 going_modular 脚本
    print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine

In [ ]:
# 设置随机种子
def set_seeds(seed: int=42):
    """为 torch 操作设置随机种子。

    参数:
        seed (int, optional): 要设置的随机种子，默认 42。
    """
    # 设置通用 torch 操作的随机种子
    torch.manual_seed(seed)
    # 设置 CUDA torch 操作（GPU 上）的随机种子
    torch.cuda.manual_seed(seed)

In [ ]:
import os
import zipfile

from pathlib import Path

import requests

def download_data(source: str, 
                  destination: str,
                  remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.
    
    Returns:
        pathlib.Path to downloaded data.
    
    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                      destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
        
        # Download pizza, steak, sushi data
        target_file = Path(source).name
        with open(data_path / target_file, "wb") as f:
            request = requests.get(source)
            print(f"[INFO] Downloading {target_file} from {source}...")
            f.write(request.content)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...") 
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
    
    return image_path

image_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                           destination="pizza_steak_sushi")
image_path

[INFO] data/pizza_steak_sushi directory exists, skipping download.


PosixPath('data/pizza_steak_sushi')

In [ ]:
from torch.utils.tensorboard import SummaryWriter
def create_writer(experiment_name: str, 
                  model_name: str, 
                  extra: str=None):
    """创建一个保存到指定 log_dir 的 SummaryWriter 实例。

    log_dir 由 runs/timestamp/experiment_name/model_name/extra 组成。

    其中 timestamp 为当前日期（YYYY-MM-DD）。

    参数:
        experiment_name (str): 实验名称。
        model_name (str): 模型名称。
        extra (str, optional): 额外目录信息，默认为 None。

    返回:
        torch.utils.tensorboard.writer.SummaryWriter(): 指向 log_dir 的 writer 实例。

    示例:
        # 创建一个写入 "runs/2022-06-04/data_10_percent/effnetb2/5_epochs/" 的 writer
        writer = create_writer(experiment_name="data_10_percent",
                               model_name="effnetb2",
                               extra="5_epochs")
    """
    from datetime import datetime
    import os

    # 获取当前日期时间戳（同一天实验放在同一目录）
    timestamp = datetime.now().strftime("%Y-%m-%d")

    if extra:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)
        
    print(f"[INFO] Created SummaryWriter, saving to: {log_dir}...")
    return SummaryWriter(log_dir=log_dir)

In [ ]:
# Create a test writer
writer = create_writer(experiment_name="test_experiment_name",
                       model_name="this_is_the_model_name",
                       extra="add_a_little_extra_if_you_want")

[INFO] Created SummaryWriter, saving to: runs/2022-06-23/test_experiment_name/this_is_the_model_name/add_a_little_extra_if_you_want...


In [ ]:
from typing import Dict, List
from tqdm.auto import tqdm

from going_modular.going_modular.engine import train_step, test_step

# 给 train() 增加 writer 参数
def train(model: torch.nn.Module, 
          train_dataloader: torch.utils.data.DataLoader, 
          test_dataloader: torch.utils.data.DataLoader, 
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device, 
          writer: torch.utils.tensorboard.writer.SummaryWriter
          ) -> Dict[str, List]:
    """训练并测试 PyTorch 模型。

    将目标模型在每个 epoch 中依次执行 train_step() 与 test_step()，
    并在训练过程中计算、打印、保存评估指标。

    若传入 writer，则将指标写入对应 log_dir。
    """
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
    }

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
        test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

        print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        if writer:
            writer.add_scalars(main_tag="Loss", 
                               tag_scalar_dict={"train_loss": train_loss,
                                                "test_loss": test_loss},
                               global_step=epoch)
            writer.add_scalars(main_tag="Accuracy", 
                               tag_scalar_dict={"train_acc": train_acc,
                                                "test_acc": test_acc}, 
                               global_step=epoch)
            writer.close()

    return results

### 下载数据

使用与 https://www.learnpytorch.io/07_pytorch_experiment_tracking/ 相同的数据。

In [ ]:
# Download 10 percent and 20 percent training data (if necessary)
data_10_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
                                     destination="pizza_steak_sushi")

data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

[INFO] data/pizza_steak_sushi directory exists, skipping download.
[INFO] data/pizza_steak_sushi_20_percent directory exists, skipping download.


In [ ]:
# Setup training directory paths
train_dir_10_percent = data_10_percent_path / "train"
train_dir_20_percent = data_20_percent_path / "train"

# Setup testing directory paths (note: use the same test dataset for both to compare the results)
test_dir = data_10_percent_path / "test"

# Check the directories
print(f"Training directory 10%: {train_dir_10_percent}")
print(f"Training directory 20%: {train_dir_20_percent}")
print(f"Testing directory: {test_dir}")

Training directory 10%: data/pizza_steak_sushi/train
Training directory 20%: data/pizza_steak_sushi_20_percent/train
Testing directory: data/pizza_steak_sushi/test


In [ ]:
from torchvision import transforms

# Create a transform to normalize data distribution to be inline with ImageNet
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], # values per colour channel [red, green, blue]
                                 std=[0.229, 0.224, 0.225])

# Create a transform pipeline
simple_transform = transforms.Compose([
                                       transforms.Resize((224, 224)),
                                       transforms.ToTensor(), # get image values between 0 & 1
                                       normalize
])

### 将数据转换为 DataLoader

In [ ]:
BATCH_SIZE = 32

# Create 10% training and test DataLoaders
train_dataloader_10_percent, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir_10_percent,
                                                                                          test_dir=test_dir,
                                                                                          transform=simple_transform,
                                                                                          batch_size=BATCH_SIZE)

# Create 20% training and test DataLoaders
train_dataloader_20_percent, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir_20_percent,
                                                                                          test_dir=test_dir,
                                                                                          transform=simple_transform,
                                                                                          batch_size=BATCH_SIZE)

# Find the number of samples/batches per dataloader (using the same test_dataloader for both experiments)
print(f"Number of batches of size {BATCH_SIZE} in 10 percent training data: {len(train_dataloader_10_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in 20 percent training data: {len(train_dataloader_20_percent)}")
print(f"Number of batches of size {BATCH_SIZE} in testing data: {len(train_dataloader_10_percent)} (all experiments will use the same test set)")
print(f"Number of classes: {len(class_names)}, class names: {class_names}")

Number of batches of size 32 in 10 percent training data: 8
Number of batches of size 32 in 20 percent training data: 15
Number of batches of size 32 in testing data: 8 (all experiments will use the same test set)
Number of classes: 3, class names: ['pizza', 'steak', 'sushi']


## 练习 1：从 [`torchvision.models`](https://pytorch.org/vision/main/models.html) 中选择一个更大的模型加入实验列表（例如 EffNetB3 或更大）

* 与现有模型相比，它的表现如何？
* **提示：** 你需要搭建与 [07. PyTorch Experiment Tracking 第 7.6 节](https://www.learnpytorch.io/07_pytorch_experiment_tracking/#76-create-experiments-and-set-up-training-code) 类似的实验流程。

In [ ]:
# TODO: your code

## 练习 2：在 20% pizza/steak/sushi 训练与测试数据上引入数据增强，对实验结果有变化吗？
    
* 例如，你可以创建一个使用数据增强的训练 DataLoader（如 `train_dataloader_20_percent_aug`）和一个不使用增强的训练 DataLoader（如 `train_dataloader_20_percent_no_aug`），然后对同一种模型在这两种 DataLoader 上的结果进行对比。
* **注意：** 你可能需要修改 `create_dataloaders()`，使其能分别接收训练集与测试集的 transform（因为测试集通常不做数据增强）。可参考 [04. PyTorch Custom Datasets 第 6 节](https://www.learnpytorch.io/04_pytorch_custom_datasets/#6-other-forms-of-transforms-data-augmentation) 的示例。
* 提示：训练集可使用 `TrivialAugmentWide` 等增强策略，测试集通常只做 resize + tensor + normalize。

In [ ]:
# TODO: your code

## 练习 3：将 FoodVision Mini 扩展为 FoodVision Big，使用完整的 [`torchvision.models` Food101 数据集](https://pytorch.org/vision/stable/generated/torchvision.datasets.Food101.html#torchvision.datasets.Food101)
    
* 你可以选择此前实验中表现最好的模型，或本 notebook 中构建的 EffNetB2 特征提取器，在完整 Food101 上训练 5 个 epoch 并观察效果。
* 如果尝试多个模型，建议记录各模型结果。
* 若从 `torchvision.models` 加载 Food101 数据集，需要先创建可用于训练的 PyTorch DataLoader。
* **注意：** Food101 的数据量远大于 pizza/steak/sushi 数据集，因此训练耗时会更长。

In [ ]:
# TODO: your code